# Session 1, Module 05: Control Flow


This module covers:
- if / elif / else with real examples
- match/case (Python 3.10+ structural pattern matching)
- Nested conditions and guard clauses
- Best practices for readable conditionals

Data Engineering Context:
Control flow is used for routing records based on type, handling different
file formats, and implementing pipeline branching logic.


## If / Elif / Else Basics


In [1]:
print("=== if / elif / else Basics ===")

# Simple if
record_count = 1000

if record_count > 0:
    print(f"Processing {record_count} records")

# if / else
is_production = True

if is_production:
    log_level = "WARNING"
else:
    log_level = "DEBUG"

print(f"Log level: {log_level}")

# if / elif / else — Multiple conditions
success_rate = 0.87

if success_rate >= 0.99:
    status = "excellent"
elif success_rate >= 0.95:
    status = "good"
elif success_rate >= 0.90:
    status = "acceptable"
else:
    status = "needs_attention"

print(f"Pipeline status: {status}")  # OUTPUT: needs_attention

=== if / elif / else Basics ===
Processing 1000 records
Log level: WARNING
Pipeline status: needs_attention


## Comparison And Logical Operators In Conditions


In [2]:
print("\n=== Complex Conditions ===")

# Multiple conditions with 'and'
batch_size = 500
memory_available_gb = 8
is_enabled = True

if batch_size <= 1000 and memory_available_gb >= 4 and is_enabled:
    print("Pipeline can run")

# Multiple conditions with 'or'
file_extension = ".csv"

if file_extension == ".csv" or file_extension == ".tsv" or file_extension == ".txt":
    print(f"Processing text file: {file_extension}")

# Better: use 'in' for membership testing
if file_extension in [".csv", ".tsv", ".txt"]:
    print(f"Processing text file (using 'in'): {file_extension}")

# Combining 'and' and 'or' — use parentheses for clarity
hour = 14
is_weekend = False
is_holiday = False

# Pipeline runs during business hours on workdays
if (9 <= hour < 17) and (not is_weekend and not is_holiday):
    print("Running during business hours")
else:
    print("Outside business hours")


=== Complex Conditions ===
Pipeline can run
Processing text file: .csv
Processing text file (using 'in'): .csv
Running during business hours


## Truthy/Falsy In Conditions


In [3]:
print("\n=== Truthy/Falsy in Conditions ===")

# Empty collections are falsy
records = []

if records:
    print(f"Processing {len(records)} records")
else:
    print("No records to process")

# Non-empty collections are truthy
records = [{"id": 1}, {"id": 2}]

if records:
    print(f"Processing {len(records)} records")

# String checks
column_name = ""

if column_name:
    print(f"Column: {column_name}")
else:
    print("No column name provided")

# CAUTION: Remember the gotcha from module 02!
count = 0  # Valid count of zero

# WRONG: This treats 0 as "not provided"
if count:
    print(f"Count: {count}")
else:
    print("Count not provided (WRONG!)")

# CORRECT: Explicitly check for None
if count is not None:
    print(f"Count: {count} (CORRECT)")


=== Truthy/Falsy in Conditions ===
No records to process
Processing 2 records
No column name provided
Count not provided (WRONG!)
Count: 0 (CORRECT)


## Nested Conditions


In [4]:
print("\n=== Nested Conditions ===")

# Sometimes nesting is unavoidable, but keep it shallow
file_type = "csv"
file_size_mb = 150
has_header = True

if file_type == "csv":
    if file_size_mb > 100:
        if has_header:
            print("Large CSV with header — using chunked reader")
        else:
            print("Large CSV without header — using chunked reader with column inference")
    else:
        print("Small CSV — loading entirely into memory")
else:
    print(f"Unsupported file type: {file_type}")


=== Nested Conditions ===
Large CSV with header — using chunked reader


## Guard Clauses — Early Exit Pattern


In [5]:
print("\n=== Guard Clauses (Early Exit) ===")

# Instead of deeply nested conditions, use early returns
def process_file(file_path: str, file_size_mb: int) -> str:
    """
    Process a file with validation.
    Uses guard clauses to handle edge cases early.
    """
    # Guard clause 1: Check for empty path
    if not file_path:
        return "Error: No file path provided"

    # Guard clause 2: Check file extension
    if not file_path.endswith((".csv", ".json", ".parquet")):
        return f"Error: Unsupported file type"

    # Guard clause 3: Check file size
    if file_size_mb > 500:
        return "Error: File too large (max 500 MB)"

    # Happy path — all validations passed
    return f"Processing {file_path} ({file_size_mb} MB)"


# Test the function
print(process_file("", 100))                      # Error: No file path
print(process_file("data.xml", 100))              # Error: Unsupported
print(process_file("data.csv", 1000))             # Error: Too large
print(process_file("data.csv", 100))              # Processing...


=== Guard Clauses (Early Exit) ===
Error: No file path provided
Error: Unsupported file type
Error: File too large (max 500 MB)
Processing data.csv (100 MB)


## Match / Case (Python 3.10+)


In [6]:
print("\n=== match / case (Python 3.10+) ===")


=== match / case (Python 3.10+) ===


Structural pattern matching — more powerful than switch/case in other languages
Basic matching

In [ ]:
def get_file_reader(file_type: str) -> str:
    """Return the appropriate reader for a file type."""
    match file_type:
        case "csv":
            return "CSVReader"
        case "json":
            return "JSONReader"
        case "parquet":
            return "ParquetReader"
        case "avro":
            return "AvroReader"
        case _:  # Default case (underscore is wildcard)
            return "UnknownReader"


print(f"Reader for csv: {get_file_reader('csv')}")
print(f"Reader for xml: {get_file_reader('xml')}")

# Matching with OR patterns
def categorize_file(extension: str) -> str:
    """Categorize file by extension."""
    match extension.lower():
        case ".csv" | ".tsv" | ".txt":
            return "text"
        case ".json" | ".jsonl":
            return "json"
        case ".parquet" | ".orc":
            return "columnar"
        case ".xlsx" | ".xls":
            return "spreadsheet"
        case _:
            return "unknown"


print(f"\n.csv category: {categorize_file('.csv')}")
print(f".parquet category: {categorize_file('.parquet')}")
print(f".xlsx category: {categorize_file('.xlsx')}")

# Matching with guards (additional conditions)
def process_record_count(count: int) -> str:
    """Determine processing strategy based on record count."""
    match count:
        case 0:
            return "skip"
        case n if n < 1000:
            return "batch_small"
        case n if n < 100000:
            return "batch_medium"
        case n if n < 1000000:
            return "batch_large"
        case _:
            return "distributed"


print(f"\n0 records: {process_record_count(0)}")
print(f"500 records: {process_record_count(500)}")
print(f"50000 records: {process_record_count(50000)}")
print(f"5000000 records: {process_record_count(5000000)}")

# Matching complex structures (tuples, lists)
def route_event(event: tuple) -> str:
    """Route an event based on its structure."""
    match event:
        case ("error", message):
            return f"Alert: {message}"
        case ("warning", message):
            return f"Log: {message}"
        case ("info", message):
            return f"Debug: {message}"
        case ("metric", name, value):
            return f"Metric {name}={value}"
        case _:
            return "Unknown event"


print(f"\n{('error', 'Connection failed')}: {route_event(('error', 'Connection failed'))}")
print(f"{('metric', 'cpu', 75)}: {route_event(('metric', 'cpu', 75))}")

# Matching dictionaries
def process_config(config: dict) -> str:
    """Process configuration based on structure."""
    match config:
        case {"type": "database", "host": host, "port": port}:
            return f"Database connection to {host}:{port}"
        case {"type": "file", "path": path}:
            return f"File source at {path}"
        case {"type": "api", "url": url}:
            return f"API endpoint: {url}"
        case _:
            return "Unknown configuration"


db_config = {"type": "database", "host": "localhost", "port": 5432}
file_config = {"type": "file", "path": "/data/input.csv"}

print(f"\n{db_config}: {process_config(db_config)}")
print(f"{file_config}: {process_config(file_config)}")

Reader for csv: CSVReader
Reader for xml: UnknownReader

.csv category: text
.parquet category: columnar
.xlsx category: spreadsheet

0 records: skip
500 records: batch_small
50000 records: batch_medium
5000000 records: distributed

('error', 'Connection failed'): Alert: Connection failed
('metric', 'cpu', 75): Metric cpu=75

{'type': 'database', 'host': 'localhost', 'port': 5432}: Database connection to localhost:5432
{'type': 'file', 'path': '/data/input.csv'}: File source at /data/input.csv


## Practical: File Format Router


In [9]:
print("\n=== Practical: File Format Router ===")

def route_file(file_path: str, file_size_mb: int) -> dict:
    """
    Determine how to process a file based on its type and size.

    Returns a configuration dict with reader and strategy.
    """
    # Extract extension
    extension = file_path.split(".")[-1].lower() if "." in file_path else ""

    # Determine reader and strategy using match/case
    match (extension, file_size_mb):
        case ("csv", size) if size < 100:
            return {"reader": "pandas", "strategy": "memory"}
        case ("csv", size) if size < 1000:
            return {"reader": "pandas", "strategy": "chunked"}
        case ("csv", _):
            return {"reader": "dask", "strategy": "distributed"}

        case ("json", size) if size < 50:
            return {"reader": "json", "strategy": "memory"}
        case ("json" | "jsonl", _):
            return {"reader": "ijson", "strategy": "streaming"}

        case ("parquet", _):
            return {"reader": "pyarrow", "strategy": "lazy"}

        case _:
            return {"reader": "unknown", "strategy": "error"}


# Test the router
test_files = [
    ("small_data.csv", 50),
    ("medium_data.csv", 500),
    ("large_data.csv", 2000),
    ("events.json", 30),
    ("events.jsonl", 200),
    ("warehouse.parquet", 5000),
    ("unknown.xyz", 100),
]

for file_path, size in test_files:
    config = route_file(file_path, size)
    print(f"{file_path:25} ({size:4} MB) → {config}")


=== Practical: File Format Router ===
small_data.csv            (  50 MB) → {'reader': 'pandas', 'strategy': 'memory'}
medium_data.csv           ( 500 MB) → {'reader': 'pandas', 'strategy': 'chunked'}
large_data.csv            (2000 MB) → {'reader': 'dask', 'strategy': 'distributed'}
events.json               (  30 MB) → {'reader': 'json', 'strategy': 'memory'}
events.jsonl              ( 200 MB) → {'reader': 'ijson', 'strategy': 'streaming'}
warehouse.parquet         (5000 MB) → {'reader': 'pyarrow', 'strategy': 'lazy'}
unknown.xyz               ( 100 MB) → {'reader': 'unknown', 'strategy': 'error'}


## Summary


In [10]:
print("\n=== Summary ===")
print("""
if / elif / else:
- Use 'in' for membership: if x in [a, b, c]
- Use parentheses for complex conditions
- Remember truthy/falsy pitfalls with 0 and ''

Guard Clauses:
- Handle edge cases early with early returns
- Keeps main logic at lower indentation level
- More readable than deeply nested conditions

match / case (Python 3.10+):
- Use _ for default/wildcard
- Use | for OR patterns: case "a" | "b"
- Add guards: case n if n > 0
- Match structures: tuples, dicts, lists
- Powerful for routing and dispatching
""")


=== Summary ===

if / elif / else:
- Use 'in' for membership: if x in [a, b, c]
- Use parentheses for complex conditions
- Remember truthy/falsy pitfalls with 0 and ''

Guard Clauses:
- Handle edge cases early with early returns
- Keeps main logic at lower indentation level
- More readable than deeply nested conditions

match / case (Python 3.10+):
- Use _ for default/wildcard
- Use | for OR patterns: case "a" | "b"
- Add guards: case n if n > 0
- Match structures: tuples, dicts, lists
- Powerful for routing and dispatching

